# K-Nearest Neighbors

## 1. Load the training data

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.base import clone
from IPython.display import display
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import ParameterSampler, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

# Find the data and preprocessing code from either notebook location.
working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'src' / 'preprocessing_pipeline.py').is_file()
     and (path / 'Dataset' / 'processed').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Run this notebook from the project root or a subfolder.')

sys.path.insert(0, str(project_root))
from src.preprocessing_pipeline import TARGET, build_preprocessor

RANDOM_STATE = 42
BAD, GOOD = 0, 1
CLASS_NAMES = ['bad', 'good']
data_dir = project_root / 'Dataset' / 'processed'

In [2]:
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')[TARGET]

print(f'Training set has {len(X_train)} rows and {X_train.shape[1]} features.')
display(y_train.value_counts().reindex([BAD, GOOD]).set_axis(CLASS_NAMES).to_frame('count'))

Training set has 700 rows and 21 features.


,count
bad,210
good,490


## 2. Tune and validate with stratified 5-fold cross-validation

Compare neighborhood sizes, distance metrics, and feature group weights. The group weights change how much the continuous, ordinal, and nominal features contribute to the distance between applicants.

Compare probability cutoffs using the same held-out rows in each fold. Select the highest mean bad-class F1, using recall and F1 variation to break ties.

### Set up the model and settings

In [3]:
baseline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', KNeighborsClassifier(n_neighbors=2, weights='uniform', metric='manhattan')),
])
param_grid = {
    'n_neighbors': [5, 7, 9, 13, 17, 23, 31, 41, 55, 75],
    'weights': ['uniform', 'distance'],
    'metric': ['manhattan', 'euclidean', 'cosine'],
    'feature_weights': [
        None,
        {'continuous': 0.5, 'ordinal': 0.5, 'nominal': 1.0},
        {'continuous': 0.5, 'ordinal': 0.25, 'nominal': 1.0},
        {'continuous': 1.0, 'ordinal': 0.5, 'nominal': 1.0},
        {'continuous': 1.0, 'ordinal': 1.0, 'nominal': 0.5},
        {'continuous': 2.0, 'ordinal': 0.5, 'nominal': 1.0},
    ],
}
candidates = [('KNN', {}, baseline)]
for params in ParameterSampler(param_grid, n_iter=96, random_state=RANDOM_STATE):
    # Group weights go in preprocessing. The rest go in KNN.
    model_params = params.copy()
    feature_weights = model_params.pop('feature_weights')
    model = clone(baseline).set_params(
        preprocessor__transformer_weights=feature_weights,
        model=KNeighborsClassifier(**model_params),
    )
    candidates.append(('KNN', params, model))

### Get validation probabilities

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))
thresholds = np.arange(15, 61, 5) / 100

# The first probability column is for bad credit (label 0).
validation_probabilities = [
    cross_val_predict(candidate, X_train, y_train, cv=folds, method='predict_proba')[:, BAD]
    for _, _, candidate in candidates
]

### Compare probability cutoffs

In [5]:
# Reuse the same predictions for each cutoff instead of fitting again.
results = []
for index, (family, _, _) in enumerate(candidates):
    bad_probability = validation_probabilities[index]
    for threshold in thresholds:
        predictions = np.where(bad_probability >= threshold, BAD, GOOD)
        fold_f1 = [
            f1_score(
                y_train.iloc[rows], predictions[rows], pos_label=BAD, zero_division=0,
            )
            for _, rows in folds
        ]
        results.append({
            'Candidate': index,
            'Model': family,
            'Threshold': threshold,
            'Mean F1 (bad)': np.mean(fold_f1),
            'F1 SD': np.std(fold_f1),
            'Recall (bad)': recall_score(y_train, predictions, pos_label=BAD, zero_division=0),
        })

### Pick the best settings

In [6]:
baseline_result = next(
    result for result in results
    if result['Candidate'] == 0 and result['Threshold'] == 0.5
)
ranked_results = pd.DataFrame(results).sort_values(
    ['Mean F1 (bad)', 'Recall (bad)', 'F1 SD'],
    ascending=[False, False, True],
)
best_result = ranked_results.iloc[0]
_, best_params, selected_model = candidates[int(best_result['Candidate'])]
best_threshold = best_result['Threshold']

In [7]:
print(f'Evaluated {len(candidates)} model settings and {len(thresholds)} thresholds.')
display(pd.DataFrame(
    [baseline_result, best_result.to_dict()], index=['Previous', 'Selected'],
).drop(columns='Candidate').round(4))

Evaluated 97 model settings and 10 thresholds.


,Model,Threshold,Mean F1 (bad),F1 SD,Recall (bad)
Previous,KNN,0.50,0.564,0.0528,0.7143
Selected,KNN,0.25,0.612,0.0333,0.8714


In [8]:
if best_params:
    with pd.option_context('display.max_colwidth', None):
        display(pd.Series(best_params, name='Selected value', dtype=object).to_frame())
else:
    print('Using the previous model settings.')

,Selected value
weights,uniform
n_neighbors,31
metric,manhattan
feature_weights,"{'continuous': 1.0, 'ordinal': 0.5, 'nominal': 1.0}"


### Fit on the full training set

In [9]:
best_model = clone(selected_model).fit(X_train, y_train)

## 3. Evaluate on the held-out test set

Compare the previous tuned model and the selected model on the same test rows. The model settings and cutoff come from cross-validation on the training set.

In [10]:
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_test = pd.read_csv(data_dir / 'y_test.csv')[TARGET]

baseline_prediction = baseline.fit(X_train, y_train).predict(X_test)
# Find the probability column for bad credit before applying the cutoff.
bad_column = list(best_model.classes_).index(BAD)
bad_probability = best_model.predict_proba(X_test)[:, bad_column]
y_pred = np.where(bad_probability >= best_threshold, BAD, GOOD)

In [11]:
rows = []
for prediction in (baseline_prediction, y_pred):
    rows.append({
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision (bad)': precision_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'Recall (bad)': recall_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'F1 (bad)': f1_score(y_test, prediction, pos_label=BAD, zero_division=0),
    })
display(pd.DataFrame(rows, index=['Previous', 'Selected']).round(4))

,Accuracy,Precision (bad),Recall (bad),F1 (bad)
Previous,0.64,0.4308,0.6222,0.5091
Selected,0.65,0.4534,0.8111,0.5817


In [12]:
print('Confusion matrix for the selected model')
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[BAD, GOOD]),
    index=pd.Index(CLASS_NAMES, name='Actual'),
    columns=pd.Index(CLASS_NAMES, name='Predicted'),
))

Confusion matrix for the selected model


Predicted,bad,good
Actual,,
bad,73,17
good,88,122


In [13]:
print(classification_report(
    y_test, y_pred, labels=[BAD, GOOD], target_names=CLASS_NAMES,
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

         bad     0.4534    0.8111    0.5817        90
        good     0.8777    0.5810    0.6991       210

    accuracy                         0.6500       300
   macro avg     0.6656    0.6960    0.6404       300
weighted avg     0.7504    0.6500    0.6639       300

